In [1]:
%pip install torch --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install librosa

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import numpy as np
import pandas as pd
import warnings
import logging

import torch
print(torch.__version__)             # Should show 2.x.x+cu121
print(torch.cuda.is_available())     # Should be True
print(torch.cuda.get_device_name(0)) # Should show GTX 1650

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import librosa

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

2.5.1+cu121
True
NVIDIA GeForce GTX 1650


In [4]:

#defaults
DURATION = 30                   
SR = 22050        #sample rate
N_MELS = 64      #num frequency bins
N_FFT = 2048      #frequency resolution - for Fourier Transform
HOP_LEN = 512     #time resolution - step size between windows

MAX_FRAMES  = 650                  # pad/truncate to this many time frames
NUM_CLASSES = 8
BATCH_SIZE  = 32
NUM_EPOCHS  = 30
LR          = 1e-3

logging.getLogger("torchaudio").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

### Load Data

In [5]:
#returns Dataframe with (filepath, genre)
def load_fma_metadata():
    
    tracks = pd.read_csv(
        os.path.join("fma_metadata" , "tracks.csv"),
        index_col=0, header=[0, 1]
    )
    small = tracks[tracks[("set", "subset")] == "small"]
    genres = small[("track", "genre_top")].dropna()
    return genres


def track_id_to_path(track_id):
    tid = f"{int(track_id):06d}"
    return os.path.join("fma_small", tid[:3], f"{tid}.mp3")

#Load an audio file and return a log-mel spectrogram as a numpy array

def audio_to_melspec(path):
    try:
        y, sr = librosa.load(path)
    except Exception:
        return np.zeros((N_MELS, MAX_FRAMES), dtype=np.float32)

    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max).astype(np.float32)

    # Pad or truncate time axis
    if log_mel.shape[1] < MAX_FRAMES:
        log_mel = np.pad(log_mel, ((0, 0), (0, MAX_FRAMES - log_mel.shape[1])))
    else:
        log_mel = log_mel[:, :MAX_FRAMES]

    # Normalise to [0, 1]
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
    return log_mel


"""
mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=SR,
    n_fft=N_FFT,
    hop_length=HOP_LEN,
    n_mels=N_MELS
)

#torchaudio
def audio_to_melspec(path):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            waveform, native_sr = torchaudio.load(path)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if native_sr != SR:
            waveform = torchaudio.transforms.Resample(native_sr, SR)(waveform)

        # Trim to duration
        max_samples = SR * DURATION
        waveform = waveform[:, :max_samples]

        # Return zeros if too short (corrupt file)
        if waveform.shape[1] < SR * 2:
            return np.zeros((N_MELS, MAX_FRAMES), dtype=np.float32)
        
        mel = mel_transform(waveform).squeeze(0).numpy()

        # Log scale
        log_mel = 10.0 * np.log10(np.maximum(mel, 1e-10)).astype(np.float32)

    except Exception:
        return np.zeros((N_MELS, MAX_FRAMES), dtype=np.float32)

    # Pad or truncate
    if log_mel.shape[1] < MAX_FRAMES:
        log_mel = np.pad(log_mel, ((0, 0), (0, MAX_FRAMES - log_mel.shape[1])))
    else:
        log_mel = log_mel[:, :MAX_FRAMES]

    # Normalise
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
    return log_mel
"""

def precompute_spectrograms(track_ids):
    os.makedirs("mel_spectrograms", exist_ok=True)
    print(f"Precomputing spectrograms for {len(track_ids)} tracks...")
    for i, tid in enumerate(track_ids):
        cache_path = os.path.join("mel_spectrograms", f"{tid}.npy")
        if os.path.exists(cache_path):
            continue  # already computed, skip
        path = track_id_to_path(tid)
        spec = audio_to_melspec(path)
        np.save(cache_path, spec)
        if i % 100 == 0:
            print(f"  {i}/{len(track_ids)} done")
    print("Precomputation complete.")

#dataset wih melspectogram and label
class FMADataset(Dataset):
    
    def __init__(self, track_ids, labels, transform=None):
        self.track_ids = track_ids
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.track_ids)

    def __getitem__(self, idx):
        cache_path = os.path.join("mel_spectrograms", f"{self.track_ids[idx]}.npy")
        spec  = np.load(cache_path)
        spec  = torch.from_numpy(spec).unsqueeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return spec, label


### Model

In [6]:
class ConvBlock(nn.Module):
    """Conv2D → BatchNorm → ReLU → MaxPool block."""
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(pool),
            nn.Dropout2d(0.2),
        )

    def forward(self, x):
        return self.block(x)


class CRNN(nn.Module):
    def __init__(self, lstm_hidden=128, lstm_layers=2, dropout=0.3):
        super().__init__()

        self.cnn = nn.Sequential(
            ConvBlock(1,   16, pool=(2, 2)),
            ConvBlock(16,  32, pool=(2, 2)),
            ConvBlock(32, 64, pool=(4, 1)),
        )

        self.freq_out = N_MELS // (2 * 2 * 4)
        self.lstm_in  = 64 * self.freq_out

        self.lstm = nn.LSTM(
            input_size=self.lstm_in,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, NUM_CLASSES),
        )

    def forward(self, x):
        x = self.cnn(x)
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T, C * F)
        x, _ = self.lstm(x)
        return self.classifier(x[:, -1, :])


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for specs, labels in tqdm(loader, desc="  Train", leave=False):
        specs, labels = specs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(specs)
        loss    = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        preds       = outputs.detach().argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for specs, labels in tqdm(loader, desc="  Eval ", leave=False):
        specs, labels = specs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs    = model(specs)
        loss       = criterion(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


### Main

In [7]:
def main():
    # ── Device setup ───────────────────────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpus = torch.cuda.device_count()
    print(f"Using device: {device}  ({n_gpus} GPU(s))")

    # ── Metadata ───────────────────────────────────────────────────────────
    print("Loading FMA metadata...")
    genre_series = load_fma_metadata()

    le = LabelEncoder()
    le.fit_transform(genre_series.values)
    class_names = le.classes_.tolist()
    print(f"Classes ({NUM_CLASSES}): {class_names}")

    tracks = pd.read_csv(
        os.path.join("fma_metadata", "tracks.csv"),
        index_col=0, header=[0, 1]
    )
    small          = tracks[tracks[("set", "subset")] == "small"].copy()
    small          = small.loc[small.index.isin(genre_series.index)]
    small["label"] = le.transform(small[("track", "genre_top")])
    small["split"] = small[("set", "split")]

    def split_data(split_name):
        subset = small[small["split"] == split_name]
        return subset.index.tolist(), subset["label"].tolist()

    train_ids, train_labels = split_data("training")
    val_ids,   val_labels   = split_data("validation")
    test_ids,  test_labels  = split_data("test")
    print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")

    #all_ids = train_ids + val_ids + test_ids
    #precompute_spectrograms(all_ids)

    # ── DataLoaders ────────────────────────────────────────────────────────
    num_workers = 0 #min(4 * max(n_gpus, 1), 8)
    pin         = device.type == "cuda"

    train_loader = DataLoader(FMADataset(train_ids, train_labels), batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=num_workers, pin_memory=pin,
                              persistent_workers=num_workers > 0)
    val_loader   = DataLoader(FMADataset(val_ids,   val_labels),   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=num_workers, pin_memory=pin,
                              persistent_workers=num_workers > 0)
    test_loader  = DataLoader(FMADataset(test_ids,  test_labels),  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=num_workers, pin_memory=pin,
                              persistent_workers=num_workers > 0)

    # ── Model ──────────────────────────────────────────────────────────────
    model = CRNN()
    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(model)

    # ── Training loop ──────────────────────────────────────────────────────
    best_val_acc = 0.0
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        train_loss, train_acc       = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss,   val_acc,  _, _  = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Train:      loss={train_loss:.4f}  acc={train_acc:.4f}")
        print(f"Validation: loss={val_loss:.4f}  acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state, "best_crnn.pth")
            print(f"  -> best model saved (val acc={best_val_acc:.4f})")

    # ── Test ───────────────────────────────────────────────────────────────
    base = model.module if isinstance(model, nn.DataParallel) else model
    base.load_state_dict(torch.load("best_crnn.pth", map_location=device))

    test_loss, test_acc, test_preds, test_labels_out = evaluate(model, test_loader, criterion, device)
    print(f"\nTest loss={test_loss:.4f}  acc={test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(test_labels_out, test_preds, target_names=class_names))


if __name__ == "__main__":
    main()


Using device: cuda  (1 GPU(s))
Loading FMA metadata...
Classes (8): ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']
Train: 6400  Val: 800  Test: 800

Model parameters: 831,016
CRNN(
  (cnn): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
        (4): Dropout2d(p=0.2, inplace=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mod

  Train:   0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 